# Sandbox Abstraction
# 0. 介绍
**研究背景**：Agent 通过工具发出的代码和命令，最终必须交给一个执行环境运行。这个环境可能是本地进程、容器、企业内网或云端沙箱，它们的调用方式和返回格式并不相同，因此外层程序需要用统一方式连接不同的执行后端。

**现存问题**：如果 Agent 的执行流程直接写死某一种后端，更换运行环境时就必须同时修改上层代码。这样，即使大模型给出了正确操作，也可能因为选错后端、调用方式不兼容或结果格式不同而无法执行；一个基础设施变化就可能让整个任务失效。

**解决方案**：本 Notebook 将实现一个极简的 Sandbox Abstraction，采用现代 Agent Runtime 常用的`统一接口 + 后端适配器`模式，用相同的 `run()`、`read()` 和 `snapshot()` 接收请求并返回标准结果，再由工厂根据配置选择合法后端，未知或不可用的后端会明确报错。然后用同一份真实 API 操作进行对比：基线版本写死后端而失败，改进版本通过统一接口选择正确后端并完成任务，从而直观看到沙箱抽象如何让 Agent 流程与具体执行基础设施解耦。
## 目录
0. 介绍
1. 初始化真实 API
2. 前置准备
3. 获取并验证 API 响应
4. 定义基线组件 *
5. 展示基线故障 *
6. 定义改进组件 *
7. 展示修复结果 *
8. 汇总消融对照

# 1. 初始化真实 API
## 连接大模型
程序需要先读取项目 `.env` 文件中已经准备好的连接信息，才能使用真实的大模型。本节直接读取这些信息并建立连接，同时保存后面要使用的模型名称。

In [1]:
from dotenv import dotenv_values, find_dotenv
from openai import OpenAI

config = dotenv_values(find_dotenv())  # 自动找到并读取项目的 .env
client = OpenAI(
    api_key=config["OPENAI_API_KEY"],
    base_url=config["OPENAI_BASE_URL"],
)
model_name = config["OPENAI_MODEL"]
print(f"真实 API 已就绪：{model_name}")

真实 API 已就绪：LongCat-2.0


输出显示了模型名称，说明真实 API 已经准备好，但此时还没有向大模型发送请求。下一章将定义大模型可以使用的工具，以及需要完成的多步任务。

# 2. 前置准备
## 2.1 说明统一的执行动作
大模型需要先知道如何提交执行请求。本节定义一个 `run_code` 工具：无论底层使用哪种执行后端，大模型都只需要填写后端名称和要运行的代码。

In [2]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "run_code",
            "description": "在指定执行后端运行代码",
            "parameters": {
                "type": "object",
                "properties": {
                    "backend": {
                        "type": "string",
                        "enum": ["python", "shell"],
                    },
                    "code": {"type": "string"},
                },
                "required": ["backend", "code"],
            },
        },
    }
]

print("可用工具：", tools[0]["function"]["name"])
print("可用后端：", tools[0]["function"]["parameters"]["properties"]["backend"]["enum"])

可用工具： run_code
可用后端： ['python', 'shell']


输出显示大模型只有一个统一工具，但可以在 `python` 和 `shell` 两个后端之间选择。此时只是说明了动作格式，还没有调用大模型或运行代码；下一步会写出具体任务。

## 2.2 写出具体任务
为了让后端选择直接影响结果，本节要求大模型使用 `python` 后端运行一行 Python 代码。相同内容如果被错误地交给 `shell`，就无法得到期望输出。

In [3]:
code_to_run = "print(6 * 7)"
messages = [
    {
        "role": "system",
        "content": "必须调用 run_code，并使用用户指定的 backend 和 code。",
    },
    {
        "role": "user",
        "content": f"请使用 python 后端运行这段代码：{code_to_run}",
    },
]

print("任务：", messages[1]["content"])

任务： 请使用 python 后端运行这段代码：print(6 * 7)


输出显示任务已经固定：后端必须是 `python`，代码必须是 `print(6 * 7)`。后面的基线和改进版本都会使用这份完全相同的任务，下一步将写出共同的成功标准。

## 2.3 定义成功标准
后端正常结束不代表任务一定完成。本节规定只有代码由 `python` 后端执行，并且标准输出等于 `42`，任务才算成功。

In [4]:
expected_backend = "python"
expected_output = "42"

print(f"成功标准：backend={expected_backend}，stdout={expected_output}")

成功标准：backend=python，stdout=42


输出显示了唯一的成功标准，说明后面的两种做法可以用同一把尺子比较。至此，统一工具、具体任务和成功标准都已准备完成；下一章将调用真实大模型，查看它选择的后端和代码。

# 3. 获取并验证 API 响应
## 3.1 获取真实响应
模型、工具和任务已经准备好，现在可以发送一次真实 API 请求。本节要求大模型必须调用 `run_code`，并记录等待回复所用的时间；这里只取得模型决定，不运行其中的代码。

In [5]:
from time import perf_counter

start = perf_counter()
response = client.chat.completions.create(
    model=model_name,
    messages=messages,
    tools=tools,
    tool_choice="required",
    temperature=0,
)
latency_ms = round((perf_counter() - start) * 1000)
raw_real = response.model_dump()

print(f"真实回复已收到：provider={config['NANO_BACKEND']}")
print(f"模型：{model_name}")
print(f"等待时间：{latency_ms} ms")

真实回复已收到：provider=openai
模型：LongCat-2.0
等待时间：3160 ms


输出显示了模型来源、模型名称和实测等待时间，说明真实 API 已经返回结果。完整响应保存在 `raw_real` 中，代码仍未执行；下一步会取出模型提交的执行请求。

## 3.2 查看模型决定
API 返回的是一份较大的数据结构，本节只取出后续执行需要的工具名称、后端和代码，同时显示停止原因与 Token 用量。这样可以先看清模型决定，再把同一决定交给两种外层程序。

In [6]:
import json

choice = raw_real["choices"][0]
tool_call = choice["message"]["tool_calls"][0]
arguments = json.loads(tool_call["function"]["arguments"])
requested_backend = arguments["backend"]
requested_code = arguments["code"]

print(f"工具：{tool_call['function']['name']}")
print(f"后端：{requested_backend}")
print(f"代码：{requested_code}")
print(f"停止原因：{choice['finish_reason']}")
print(f"Token 用量：{raw_real['usage']}")

工具：run_code
后端：python
代码：print(6 * 7)
停止原因：tool_calls
Token 用量：{'completion_tokens': 74, 'prompt_tokens': 188, 'total_tokens': 262, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 43, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 128, 'image_tokens': 0, 'video_tokens': 0, 'text_tokens': 0}, 'effectiveCachedTokens': 128, 'cache_write_tokens': 0, 'cache_read_tokens': 0, 'input_tokens': 0, 'output_tokens': 0, 'output_tokens_details': None, 'cached_tokens': 0}


输出显示大模型选择了 `run_code`，并给出了任务要求的 `python` 后端和 Python 代码，说明模型已经做出正确决定。停止原因表示模型正在等待外层程序执行工具，Token 用量记录了本次请求的实际消耗；下一章将定义一个写死执行后端的基线组件。

# 4. 定义基线组件
## 4.1 定义写死后端的执行器
最直接的执行程序会把所有内容固定交给一种后端。本节保留这种做法作为基线：函数只会调用 `shell`，没有接收后端参数，因此第 3 章模型选择的 `python` 无法传到执行层。

In [7]:
import subprocess


def run_with_fixed_shell(code):
    # 基线把所有内容固定交给 Shell，不读取模型选择的后端
    completed = subprocess.run(
        ["bash", "-lc", code],
        capture_output=True,
        text=True,
    )
    return {
        "backend": "shell",
        "stdout": completed.stdout.strip(),
        "stderr": completed.stderr.strip(),
        "returncode": completed.returncode,
    }


print("基线组件已定义：所有内容固定交给 shell")

基线组件已定义：所有内容固定交给 shell


输出说明基线组件已经定义完成，但还没有运行模型给出的代码。它只认识 `shell`，也没有位置接收 `requested_backend`；下一章将把第 3 章保存的同一份真实模型决定交给它，观察任务结果。

# 5. 展示基线故障
## 5.1 运行真实模型给出的代码
第 3 章的真实模型决定要求使用 `python`，但基线执行器只会使用 `shell`。本节把同一段模型代码交给基线，并同时显示模型要求、实际后端和执行结果，从而看清数据在哪里发生偏离。

In [8]:
# 把同一份真实模型代码交给写死 Shell 的基线
baseline_result = run_with_fixed_shell(requested_code)

print(f"模型要求：{requested_backend}")
print(f"实际使用：{baseline_result['backend']}")
print(f"代码：{requested_code}")
print(f"标准输出：{baseline_result['stdout']}")
print(f"错误输出：{baseline_result['stderr']}")
print(f"退出码：{baseline_result['returncode']}")

模型要求：python
实际使用：shell
代码：print(6 * 7)
标准输出：
错误输出：bash: -c: line 0: syntax error near unexpected token `6'
bash: -c: line 0: `print(6 * 7)'
退出码：2


输出显示模型要求 `python`，基线却实际使用了 `shell`。Shell 无法把 `print(6 * 7)` 当作 Python 代码运行，因此没有产生 `42`，并返回了非零退出码；下一步将用统一成功标准给出明确结论。

## 5.2 判断基线结果
执行失败还需要落到任务结果上。本节使用第 2 章已经固定的同一标准：实际后端必须是 `python`，标准输出必须是 `42`。

In [9]:
# 使用第 2 章的同一标准判断基线是否完成任务
baseline_success = (
    baseline_result["backend"] == expected_backend
    and baseline_result["stdout"] == expected_output
)

print(f"基线任务成功：{baseline_success}")

基线任务成功：False


输出为 `False`，说明基线没有完成任务。大模型已经正确选择 `python` 并给出正确代码，故障来自外层程序写死了 `shell`；下一章将定义能够接收后端参数并选择对应执行方式的 Sandbox Abstraction。

# 6. 定义改进组件
## 6.1 定义 Python Adapter
改进的关键不是让上层代码认识每种执行细节，而是让每个后端接收相同输入并返回相同结构。现有 Shell 执行器已经接收 `code`，本节新增一个同样接收 `code` 的 Python Adapter，并返回相同的四项结果。

In [10]:
import sys


def run_with_python(code):
    # Adapter 把统一的 code 输入转换成 Python 解释器命令
    completed = subprocess.run(
        [sys.executable, "-c", code],
        capture_output=True,
        text=True,
    )
    return {
        "backend": "python",
        "stdout": completed.stdout.strip(),
        "stderr": completed.stderr.strip(),
        "returncode": completed.returncode,
    }


print("Python Adapter 已定义")

Python Adapter 已定义


输出说明 Python Adapter 已经定义完成，但还没有运行代码。它与 Shell 执行器使用相同输入和返回字段，因此上层程序不需要理解两种命令格式；下一步把它们登记到同一个后端表中。

## 6.2 注册可用后端
程序还需要知道后端名称对应哪个 Adapter。本节用一张简单的表完成登记：`python` 指向 Python Adapter，`shell` 指向已有的 Shell 执行器。

In [11]:
# Registry 只保存后端名称与 Adapter 的对应关系
sandboxes = {
    "python": run_with_python,
    "shell": run_with_fixed_shell,
}

print("已注册后端：", list(sandboxes))

已注册后端： ['python', 'shell']


输出显示两个后端已经拥有稳定名称。以后增加新后端时，只需增加对应 Adapter 并登记，不需要改动模型任务或上层执行流程；下一步定义唯一的调用入口。

## 6.3 定义统一入口
有了后端表，上层程序只需传入模型给出的 `backend` 和 `code`。本节定义统一的 `run_with_abstraction`，先按名称取出 Adapter，再用相同方式运行代码。

In [12]:
def run_with_abstraction(backend, code):
    # 统一入口根据模型给出的名称选择对应 Adapter
    sandbox = sandboxes[backend]
    return sandbox(code)


print("统一入口已定义：run_with_abstraction(backend, code)")

统一入口已定义：run_with_abstraction(backend, code)


输出说明 Sandbox Abstraction 已经定义完成，但本章仍未运行任务。上层程序现在只依赖一个统一入口，具体命令格式留在各自 Adapter 内部；下一章将把与基线完全相同的真实模型决定交给这个入口。

# 7. 展示修复结果
## 7.1 通过统一入口运行代码
为了只比较有无 Sandbox Abstraction，本节继续使用第 3 章保存的 `requested_backend` 和 `requested_code`。统一入口会读取模型选择的后端，再把代码交给对应 Adapter。

In [13]:
# 把同一份真实模型决定交给统一入口
improved_result = run_with_abstraction(
    requested_backend,
    requested_code,
)

print(f"模型要求：{requested_backend}")
print(f"实际使用：{improved_result['backend']}")
print(f"代码：{requested_code}")
print(f"标准输出：{improved_result['stdout']}")
print(f"错误输出：{improved_result['stderr']}")
print(f"退出码：{improved_result['returncode']}")

模型要求：python
实际使用：python
代码：print(6 * 7)
标准输出：42
错误输出：
退出码：0


输出显示模型要求和实际使用的后端都是 `python`。Python Adapter 正确运行了 `print(6 * 7)`，标准输出为 `42`，退出码为 0；下一步仍用与基线完全相同的标准判断任务结果。

## 7.2 判断改进结果
执行结果已经出现，还需要落到任务是否完成。本节仍然要求实际后端等于 `python`、标准输出等于 `42`，没有改变成功标准。

In [14]:
# 使用第 2 章的同一标准判断改进版本是否完成任务
improved_success = (
    improved_result["backend"] == expected_backend
    and improved_result["stdout"] == expected_output
)

print(f"改进任务成功：{improved_success}")

改进任务成功：True


输出为 `True`，说明改进版本完成了任务。模型、代码和成功标准都没有改变，唯一变化是外层程序读取了模型选择的后端，并通过统一接口调用对应 Adapter；下一章将汇总两种做法的完整对照。

# 8. 汇总消融对照
## 8.1 对比两种做法
两种做法使用同一次真实 API 调用、同一份模型决定和同一个成功标准。本节把共同输入与运行指标放在一起，再并排记录实际后端、输出、退出码和任务结果，从而确认差异只来自有无 Sandbox Abstraction。

In [15]:
# 汇总同一次真实模型决定下的基线与改进结果
comparison = {
    "共同输入": {
        "provider": config["NANO_BACKEND"],
        "model": model_name,
        "API 调用次数": 1,
        "Token": raw_real["usage"]["total_tokens"],
        "等待时间_ms": latency_ms,
        "模型选择后端": requested_backend,
        "模型给出代码": requested_code,
    },
    "基线": {
        "实际后端": baseline_result["backend"],
        "标准输出": baseline_result["stdout"],
        "退出码": baseline_result["returncode"],
        "任务成功": baseline_success,
    },
    "改进": {
        "实际后端": improved_result["backend"],
        "标准输出": improved_result["stdout"],
        "退出码": improved_result["returncode"],
        "任务成功": improved_success,
    },
}

print(json.dumps(comparison, indent=2, ensure_ascii=False))

{
  "共同输入": {
    "provider": "openai",
    "model": "LongCat-2.0",
    "API 调用次数": 1,
    "Token": 262,
    "等待时间_ms": 3160,
    "模型选择后端": "python",
    "模型给出代码": "print(6 * 7)"
  },
  "基线": {
    "实际后端": "shell",
    "标准输出": "",
    "退出码": 2,
    "任务成功": false
  },
  "改进": {
    "实际后端": "python",
    "标准输出": "42",
    "退出码": 0,
    "任务成功": true
  }
}


输出显示两种做法共享同一次真实模型调用：基线写死 `shell`，没有得到标准输出，任务失败；改进版本读取模型选择的 `python`，得到 `42`，任务成功。模型和代码都没有改变，决定结果的是外层程序能否通过统一接口把请求交给正确后端，至此本 Notebook 的对照实验结束。

## 8.2 拓展

### nano 版省略了什么

nano 版只在 Python 与 shell 两个 Adapter 之间选择，没有统一文件、网络、资源、身份、日志、快照与产物接口，也没有声明各后端不支持的 capability。生产抽象应提供能力探测和显式降级错误，避免同一调用在不同隔离后端产生悄然不同的安全语义。

### 延伸阅读

1. 2025, [Docker, Docker Sandboxes: A New Approach for Coding Agent Safety](https://www.docker.com/blog/docker-sandboxes-a-new-approach-for-coding-agent-safety/)：为编码 Agent 提供隔离工作区与统一运行边界。
2. 2025, [Anthropic, Beyond permission prompts](https://www.anthropic.com/engineering/claude-code-sandboxing)：文件与网络边界如何减少逐次权限提示。
3. 2024, [SWE-agent: Agent-Computer Interfaces Enable Automated Software Engineering](https://arxiv.org/abs/2405.15793)：执行接口设计对 Agent 行为和迁移性的影响。